In [ ]:
!pip install python-igraph
!git clone https://github.com/PayamDiba/SERGIO.git

import sys
sys.path.append('/content/SERGIO')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 78.3 MB/s eta 0:00:00
Cloning into 'SERGIO'...
remote: Enumerating objects: 636, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 636 (delta 24), reused 23 (delta 23), pack-reused 602 (from 1)
Receiving objects: 100% (636/636), 898.08 MiB | 23.59 MiB/s, done.
Resolving deltas: 100% (160/160), done.
Updating files: 100% (324/324), done.


In [ ]:
import numpy as np
import random

# Network structure parameters
n_MR = 5
n_TF = 100
n_G = 100

targets_filename = "targets_file.txt"

with open(targets_filename, "w") as f:
    # 1. TFs regulated by MRs (Layer 1 -> Layer 2)
    # The paper sets the number of incoming edges to TFs (d_MR->TF) to 3
    d_mr_tf = 3
    for tf_id in range(n_MR, n_MR + n_TF):
        regs = random.sample(range(0, n_MR), d_mr_tf)

        # Each K was sampled from a uniform distribution on [1.0, 5.0] and negated with 0.2 probability [cite: 79]
        ks = [np.random.uniform(1.0, 5.0) * (-1 if random.random() < 0.2 else 1) for _ in range(d_mr_tf)]

        # Each Hill coefficient was set to 2 with 0.9 probability and 1 with 0.1 probability [cite: 80]
        hills = [2 if random.random() < 0.9 else 1 for _ in range(d_mr_tf)]

        line = f"{tf_id},{d_mr_tf}," + ",".join(map(str, regs)) + "," + ",".join(map(str, ks)) + "," + ",".join(map(str, hills)) + "\n"
        f.write(line)

    # 2. Target Genes regulated by TFs (Layer 2 -> Layer 3)
    # The paper sets the incoming edges to Target Genes (d_TF->G) in the range [3, 7]
    for g_id in range(n_MR + n_TF, n_MR + n_TF + n_G):
        d_tf_g = random.randint(3, 7)
        regs = random.sample(range(n_MR, n_MR + n_TF), d_tf_g)

        ks = [np.random.uniform(1.0, 5.0) * (-1 if random.random() < 0.2 else 1) for _ in range(d_tf_g)]
        hills = [2 if random.random() < 0.9 else 1 for _ in range(d_tf_g)]

        line = f"{g_id},{d_tf_g}," + ",".join(map(str, regs)) + "," + ",".join(map(str, ks)) + "," + ",".join(map(str, hills)) + "\n"
        f.write(line)

print(f"Successfully generated {targets_filename}")

Successfully generated targets_file.txt


In [ ]:
mr_filename = "master_regulators_file.txt"
n_cell_types = 100

# Arbitrary low and high ranges for basal production
low_range = (0.1, 0.5)
high_range = (1.5, 3.0)

with open(mr_filename, "w") as f:
    for mr_id in range(n_MR):
        rates = []
        for _ in range(n_cell_types):
            # Sample uniformly from either the low or high range for each condition
            if random.random() < 0.5:
                rate = np.random.uniform(*low_range)
            else:
                rate = np.random.uniform(*high_range)
            rates.append(rate)

        line = f"{mr_id}," + ",".join(map(str, rates)) + "\n"
        f.write(line)

print(f"Successfully generated {mr_filename}")

Successfully generated master_regulators_file.txt


In [ ]:
import numpy as np
import sys
import importlib

# Patch the SERGIO library to fix the deprecated np.int and np.float usage
sergio_path = '/content/SERGIO/SERGIO/sergio.py'
print(f"Patching file: {sergio_path}")

original_content = ""
with open(sergio_path, 'r') as f:
    original_content = f.read()

# Apply patches for deprecated numpy types
patched = False
content = original_content

if "np.int(" in content:
    print("Found 'np.int(' in the file. Applying patch...")
    content = content.replace('np.int(', 'int(')
    patched = True

if "np.float(" in content:
    print("Found 'np.float(' in the file. Applying patch...")
    content = content.replace('np.float(', 'float(')
    patched = True

if patched:
    with open(sergio_path, 'w') as f:
        f.write(content)
    print("Patch applied.")
else:
    print(" 'np.int(' and 'np.float(' not found or already patched.")

# Before importing, remove the module from sys.modules if it exists to ensure a fresh load
if 'SERGIO.sergio' in sys.modules:
    print("SERGIO.sergio module already loaded. Removing from sys.modules to force reload.")
    del sys.modules['SERGIO.sergio']

from SERGIO.sergio import sergio

total_genes = n_MR + n_TF + n_G # 5 + 100 + 100 = 205

# Initialize SERGIO
# (noise_params and decays are arbitrary basal settings required by the simulator)
sim = sergio(number_genes=total_genes,
             number_bins=n_cell_types,
             number_sc=1,
             noise_params=1,
             noise_type='dpd',
             decays=0.8)

# Build Graph from our generated files
print("Building network graph...")
sim.build_graph(targets_filename, mr_filename, shared_coop_state=2)

# Run Simulation
print("Running simulation (this may take a moment)...")
sim.simulate()

# Retrieve and format the Expression Matrix
expr_matrix_list = sim.getExpressions()

# Convert the output list to a 2D numpy array (Genes/Rows x Conditions/Columns)
expr_matrix = np.concatenate(expr_matrix_list, axis=1)

print("Simulation Complete!")
print(f"Expression Matrix Shape: {expr_matrix.shape}")
# Expected shape: (205, 100)

Patching file: /content/SERGIO/SERGIO/sergio.py
Found 'np.float(' in the file. Applying patch...
Patch applied.
SERGIO.sergio module already loaded. Removing from sys.modules to force reload.
Building network graph...
Running simulation (this may take a moment)...
Start simulating new level
There are 5 genes to simulate in this layer
Done with current level
Start simulating new level
There are 100 genes to simulate in this layer
Done with current level
Start simulating new level
There are 100 genes to simulate in this layer
Done with current level
Simulation Complete!
Expression Matrix Shape: (205, 100)


In [ ]:
import pandas as pd

# Create descriptive labels for rows (Genes) and columns (Conditions)
gene_labels = [f"Gene_{i}" for i in range(total_genes)]
condition_labels = [f"Condition_{i}" for i in range(n_cell_types)]

# Convert the numpy array to a Pandas DataFrame
df_expr = pd.DataFrame(expr_matrix, index=gene_labels, columns=condition_labels)

# Save the DataFrame to a CSV file
csv_filename = "simulated_expression_matrix.csv"
df_expr.to_csv(csv_filename)

print(f"Expression matrix successfully saved to {csv_filename}!")
print("\nFirst 5 rows and 5 columns preview:")
print(df_expr.iloc[:5, :5])

Expression matrix successfully saved to simulated_expression_matrix.csv!

First 5 rows and 5 columns preview:
        Condition_0  Condition_1  Condition_2  Condition_3  Condition_4
Gene_0     3.362785     0.218883     0.186280     0.443588     0.785054
Gene_1     2.744995     0.174296     0.522184     0.689546     2.836243
Gene_2     0.622029     0.584753     0.164918     0.140191     0.088402
Gene_3     4.305445     0.246788     0.016797     0.542229     1.759619
Gene_4     3.008680     1.355408     0.360740     0.366743     0.133935


In [ ]:
import numpy as np
import pandas as pd

# Network parameters (same as before)
n_MR = 5
n_TF = 100
n_G = 100

# Initialize a binary matrix of zeros (Target Genes x Transcription Factors)
# Shape will be (100, 100)
ground_truth_matrix = np.zeros((n_G, n_TF), dtype=int)

# Read the targets file we generated in Step 2
with open("targets_file.txt", "r") as f:
    lines = f.readlines()

for line in lines:
    parts = line.strip().split(',')
    target_node_id = int(parts[0])
    num_regulators = int(parts[1])

    # We only care about Target Genes (Layer 3) being regulated by TFs (Layer 2)
    # Target gene IDs start at (n_MR + n_TF), which is 105
    if target_node_id >= (n_MR + n_TF):

        # Shift the index so Gene 105 becomes row index 0
        gene_idx = target_node_id - (n_MR + n_TF)

        # The regulators are listed immediately after the num_regulators
        regulator_ids = [int(x) for x in parts[2:2+num_regulators]]

        for reg_id in regulator_ids:
            # Shift the index so TF 5 becomes column index 0
            tf_idx = reg_id - n_MR

            # Mark the edge as 1 (True connection)
            ground_truth_matrix[gene_idx, tf_idx] = 1

# Convert to a Pandas DataFrame for easy viewing and saving
gene_names = [f"TargetGene_{i}" for i in range(n_G)]
tf_names = [f"TF_{i}" for i in range(n_TF)]

df_labels = pd.DataFrame(ground_truth_matrix, index=gene_names, columns=tf_names)

# Save to CSV
labels_filename = "ground_truth_labels.csv"
df_labels.to_csv(labels_filename)

print(f"Ground truth labels successfully saved to {labels_filename}!")
print("\nPreview of the first 5 genes and their connections to the first 10 TFs:")
print(df_labels.iloc[:5, :10])

# Quick sanity check: Print total number of actual edges
total_edges = np.sum(ground_truth_matrix)
print(f"\nTotal number of true regulatory edges in this matrix: {total_edges}")

Ground truth labels successfully saved to ground_truth_labels.csv!

Preview of the first 5 genes and their connections to the first 10 TFs:
              TF_0  TF_1  TF_2  TF_3  TF_4  TF_5  TF_6  TF_7  TF_8  TF_9
TargetGene_0     0     0     0     0     0     0     0     0     0     0
TargetGene_1     0     0     0     0     0     0     0     0     0     0
TargetGene_2     1     0     0     0     0     0     0     0     0     0
TargetGene_3     0     0     1     0     0     0     0     0     0     0
TargetGene_4     0     0     0     0     0     0     0     0     0     0

Total number of true regulatory edges in this matrix: 507
